# PR Guardian — QLoRA Training Session

Fine-tunes **Qwen2.5-Coder-1.5B-Instruct** into a **code-review bug detector** on the PR Guardian dataset, evaluates it on a held-out **per-repo test split**, and zips the LoRA adapter for download.

**Before running:**
- Settings → Accelerator → **GPU (T4)**
- Settings → Internet → **ON** (only needed for the base model + optional Devign fallback)
- Add Input → your **pr-guardian-diffs** dataset (the mined `*.jsonl`). If you attach nothing, the notebook falls back to the public Devign seed dataset so it still runs end-to-end.

Pipeline: load mined diffs → repo-level train/val/test split + de-dup → QLoRA (NF4, r=16) → train → evaluate (accuracy / macro-F1 / PR-AUC) → save adapter.


In [ ]:
!pip install -q peft bitsandbytes

import torch

assert torch.cuda.is_available(), 'Enable GPU first: Settings -> Accelerator -> GPU T4'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM(GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
from pathlib import Path

# ---- CONFIG ----------------------------------------------------------------
BASE_MODEL  = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'   # fallback: Qwen/Qwen2.5-Coder-0.5B-Instruct
TASK        = 'verdict'                            # 'verdict' (BUG/CLEAN) or 'review' (full JSON)
EPOCHS      = 2.0
LR          = 2e-4
MAX_LEN     = 2048
BATCH       = 1
GRAD_ACCUM  = 16
LORA_R      = 16
SEED        = 7
WORK        = Path('/kaggle/working')
# ---------------------------------------------------------------------------

INPUT_JSONL = sorted(Path('/kaggle/input').rglob('*.jsonl'))
print('attached jsonl files:', [str(p) for p in INPUT_JSONL])

In [ ]:
import json
import requests

def load_guardian_records(paths):
    recs = []
    for p in paths:
        for line in p.read_text(encoding='utf-8', errors='ignore').splitlines():
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            if {'kind', 'diff'}.issubset(r.keys()):
                recs.append(r)
    return recs

records = load_guardian_records(INPUT_JSONL)

if not records:  # fallback: public Devign/CodeXGLUE seed (function-level C, buggy/clean)
    print('No PR-Guardian JSONL attached — downloading the Devign seed (Internet must be ON)')
    CODEXGLUE = 'https://raw.githubusercontent.com/microsoft/CodeXGLUE/main/Code-Code-Defect-Detection/dataset/{}.jsonl'
    rows = []
    for split in ('train', 'valid', 'test'):
        text = requests.get(CODEXGLUE.format(split), timeout=120).text
        rows += [dict(r, _split=split) for r in map(json.loads, text.splitlines()) if r]
    records = [
        {
            'id': f"devign_{r['_split']}_{i}",
            'repo': f"devign/{r['_split']}",
            'kind': 'bug' if int(r['target']) == 1 else 'clean',
            'path': f'function_{i}.c',
            'lang': 'C',
            'diff': '\n'.join('+' + ln for ln in r['func'].splitlines()),
        }
        for i, r in enumerate(rows)
    ]

n_bug = sum(r['kind'] == 'bug' for r in records)
print(f'{len(records)} records | {n_bug} bugs | {len(records)-n_bug} clean | repos: {len({r["repo"] for r in records})}')

In [ ]:
import hashlib
import re

# ---- prompts (identical to training/prompts.py in the repo) -----------------
SYSTEM = (
    'You are a meticulous senior code reviewer. You examine code diffs for bugs '
    'that will break behavior at runtime: null/None hazards, index errors, '
    'resource leaks, unhandled exceptions, security issues, concurrency bugs, '
    'API misuse, and logic errors. Style, naming, and refactoring suggestions '
    'are out of scope. Be precise: report BUG only when the diff itself '
    'introduces a real defect. '
    'Respond ONLY with a JSON object in this exact shape: {"verdict": "BUG" or "CLEAN", '
    '"type": one of null_safety, index_error, resource_leak, exception_handling, '
    'security, concurrency, api_misuse, logic, other, "severity": one of low, medium, '
    'high, critical, "comment": short description of the problem (null when CLEAN), '
    '"location": file path or null, "confidence": number 0.0-1.0}. '
    'Output no text other than the JSON object.'
)

def build_example(rec):
    diff = rec['diff'][:6000]
    user = f'Review the following diff and decide whether it introduces a bug.\n\nDiff:\n```diff\n{diff}\n```'
    if rec['kind'] == 'bug':
        if TASK == 'verdict':
            answer = '{"verdict": "BUG", "confidence": 0.9}'
        else:
            comment = (rec.get('review_comment') or rec.get('fix_msg') or 'This change introduces a defect.')[:400]
            answer = json.dumps({'verdict': 'BUG', 'type': rec.get('bug_type') or 'other',
                                 'severity': rec.get('severity') or 'medium', 'comment': comment,
                                 'location': rec.get('path'), 'confidence': 0.9})
    else:
        answer = '{"verdict": "CLEAN", "confidence": 0.9}' if TASK == 'verdict' else \
                 '{"verdict": "CLEAN", "comment": null, "location": null, "confidence": 0.9}'
    return {'messages': [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': answer},
    ], 'meta': {'id': rec['id'], 'repo': rec['repo'], 'kind': rec['kind']}}

# ---- exact dedup ------------------------------------------------------------
def norm(diff):
    lines = [l.strip() for l in diff.lower().splitlines()
             if not l.startswith(('+++', '---', '@@', 'diff --git', 'index '))]
    return '\n'.join(lines)

seen, unique = set(), []
for rec in records:
    h = hashlib.sha1(norm(rec['diff']).encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        unique.append(rec)
print(f'exact dedup: {len(records)} -> {len(unique)}')
records = unique

# ---- repo-level split (a repo never straddles train and test) -----------------
repos = sorted({r['repo'] for r in records})
ordered = sorted(repos, key=lambda x: hashlib.md5(x.encode()).hexdigest())
n_test = max(1, round(len(ordered) * 0.1)) if len(ordered) >= 3 else 0
n_val = max(1, round(len(ordered) * 0.1)) if len(ordered) >= 3 else 0
assign = {}
for i, repo in enumerate(ordered):
    assign[repo] = 'test' if i < n_test else ('val' if i < n_test + n_val else 'train')
if len(repos) < 3:  # few repos: honest fallback, record-level split
    import random
    random.Random(SEED).shuffle(records)
    n = len(records)
    splits = {'test': records[:max(1, round(n*0.1))], 'val': records[max(1, round(n*0.1)):max(1, round(n*0.2))],
              'train': records[max(1, round(n*0.2)):]}
else:
    splits = {'train': [], 'val': [], 'test': []}
    for rec in records:
        splits[assign[rec['repo']]].append(rec)

data = {k: [build_example(r) for r in v] for k, v in splits.items()}
print({k: len(v) for k, v in data.items()})
for k, v in data.items():
    print(k, 'bugs:', sum(x['meta']['kind'] == 'bug' for x in v), '/', len(v))

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map={'': 0})
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
model.print_trainable_parameters()
model.config.use_cache = False

def tokenize_example(ex):
    prompt = tok.apply_chat_template(ex['messages'][:-1], tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(ex['messages'], tokenize=False)
    p_ids = tok(prompt, add_special_tokens=False)['input_ids']
    f_ids = tok(full, add_special_tokens=False)['input_ids']
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    if len(f_ids) > MAX_LEN:
        f_ids, labels = f_ids[-MAX_LEN:], labels[-MAX_LEN:]
    return {'input_ids': f_ids, 'labels': labels, 'attention_mask': [1] * len(f_ids)}

import torch.utils.data as D

class ChatDS(D.Dataset):
    def __init__(self, items):
        self.items = [tokenize_example(x) for x in items]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    m = max(len(b['input_ids']) for b in batch)
    pad = tok.pad_token_id
    return {'input_ids': torch.tensor([b['input_ids'] + [pad] * (m - len(b['input_ids'])) for b in batch]),
            'labels': torch.tensor([b['labels'] + [-100] * (m - len(b['labels'])) for b in batch]),
            'attention_mask': torch.tensor([b['attention_mask'] + [0] * (m - len(b['attention_mask'])) for b in batch])}

train_ds = ChatDS(data['train'])
val_ds = ChatDS(data['val']) if data['val'] else None
print('tokenized:', len(train_ds), 'train examples')

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=str(WORK / 'ckpt'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=5,
    save_strategy='no',
    report_to=[],
    bf16=torch.cuda.is_bf16_supported(),
    optim='paged_adamw_8bit',
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collate)
trainer.train()

losses = [(l['step'], l['loss']) for l in trainer.state.log_history if 'loss' in l]
if losses:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 3))
    plt.plot([s for s, _ in losses], [l for _, l in losses])
    plt.xlabel('step'); plt.ylabel('train loss'); plt.title('QLoRA training loss'); plt.show()

In [ ]:
from sklearn.metrics import average_precision_score, f1_score

model.config.use_cache = True
model.eval()

def generate(messages_list, max_new_tokens=96, bs=8):
    out = []
    tok.padding_side = 'left'
    for i in range(0, len(messages_list), bs):
        batch = messages_list[i:i + bs]
        texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch]
        enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        out += [tok.decode(g[j][enc['input_ids'].shape[1]:], skip_special_tokens=True) for j in range(len(batch))]
    tok.padding_side = 'right'
    return out

JSON_RE = re.compile(r'\{.*\}', re.DOTALL)

def parse(text):
    m = JSON_RE.search(text)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

test_ex = data['test'][:200]
outputs = generate([ex['messages'][:-1] for ex in test_ex])
parsed = [parse(o) for o in outputs]
y_true = ['BUG' if ex['meta']['kind'] == 'bug' else 'CLEAN' for ex in test_ex]
y_pred = [(p['verdict'] if isinstance(p, dict) and p.get('verdict') in ('BUG', 'CLEAN') else 'INVALID') for p in parsed]
conf = [float(p.get('confidence', 0)) if isinstance(p, dict) else None for p in parsed]

results = {
    'task': TASK, 'base_model': BASE_MODEL, 'n_test': len(test_ex),
    'valid_output_rate': round(sum(p is not None for p in parsed) / len(test_ex), 4),
    'accuracy': round(sum(t == p for t, p in zip(y_true, y_pred)) / len(test_ex), 4),
    'macro_f1': round(f1_score(y_true, y_pred, labels=['BUG', 'CLEAN'], average='macro', zero_division=0), 4),
}
if len(set(y_true)) == 2 and all(c is not None for c in conf):
    results['pr_auc'] = round(average_precision_score([t == 'BUG' for t in y_true], conf), 4)

print(json.dumps(results, indent=2))
print('\n--- sample outputs ---')
for ex, o in list(zip(test_ex, outputs))[:3]:
    print('true:', ex['meta']['kind'], '| out:', o.replace(chr(10), ' ')[:160])

In [ ]:
# ---- save adapter + results for download -------------------------------------
ADAPTER = 'pr-guardian-' + TASK + '-v1'
model.save_pretrained(WORK / ADAPTER)
tok.save_pretrained(WORK / ADAPTER)
(WORK / ADAPTER / 'results.json').write_text(json.dumps(results, indent=2))
(WORK / ADAPTER / 'kaggle_run_meta.json').write_text(json.dumps(
    {'base_model': BASE_MODEL, 'epochs': EPOCHS, 'lr': LR, 'lora_r': LORA_R,
     'max_len': MAX_LEN, 'grad_accum': GRAD_ACCUM, 'seed': SEED,
     'n_train': len(train_ds), 'n_test': len(test_ex), 'loss_history': losses[-20:]}, indent=2))

!cd /kaggle/working && zip -qr {ADAPTER}.zip {ADAPTER}
print('done ->', WORK / (ADAPTER + '.zip'))
print('\nNext: download the zip, unzip to models/' + TASK + '-v1/ locally, then run the API/CLI with')
print('PRGUARDIAN_MOCK=0 ADAPTER_PATH=models/' + TASK + '-v1')

## After this run

1. Download `pr-guardian-verdict-v1.zip` from the notebook Output panel (or run a version of this notebook via `kaggle kernels push` and pull the output).
2. Unzip to `models/verdict-v1/` in your local PR Guardian clone.
3. Locally: `PRGUARDIAN_MOCK=0 ADAPTER_PATH=models/verdict-v1 .venv/bin/python -m backend.cli.prguard review --diff my.diff`
4. Baseline comparison: `python -m eval.harness --systems api ollama finetuned --adapter models/verdict-v1`

OOM? Switch to `Qwen/Qwen2.5-Coder-0.5B-Instruct`, drop `MAX_LEN` to 1024, set `LORA_R = 8`.
